In [1]:
# cargar modelo

In [ ]:
import os
os.chdir('mmsegmentation')

In [ ]:
# =========================
# 1) Configuración general
# =========================
import os, glob, csv, math
import numpy as np
import torch
import torch.nn.functional as F
import cv2

# --- TUS RUTAS ---
CFG_FILE  = 'configs/MobileNet/Mobilenet_test.py'
CKPT_FILE = "work_dirs/checkpoint.pth"
IMG_DIR   = "carpeta secuencia"
OUT_CSV   = "output.csv"

# Qué salida del backbone usar
FEAT_SOURCE = "backbone_out0"  # "backbone_out0" | "backbone_out1" | "backbone_out2"
FEAT_INDEX  = {"backbone_out0": 0, "backbone_out1": 1, "backbone_out2": 2}[FEAT_SOURCE]

# Parámetros de PC
DRIVABLE_ID = 1      # id de la clase "carretera" en tu modelo (cámbialo si es 0)
R_FIXED     = 12     # <<< R FIJO (elige 8/12/16/24, etc). NO se usa tiempo.
KMAX        = 17     # ventana máxima => R' <= 8 (si R_FIXED > 8 se hace downsample)
TILE        = 64     # bloque espacial para evitar OOM
USE_FP16    = True   # usa FP16 si hay CUDA (ahorra memoria)
DS_MIN      = 1      # downsample mínimo de features

# Submuestreo opcional dentro del directorio (sin tiempos):
#   SKIP = 1 -> pares (0,1), (1,2), (2,3) ...
#   SKIP = 2 -> pares (0,2), (1,3), (2,4) ...
SKIP        = 1

# Limitar nº de pares (útil para pruebas). None = todos.
MAX_PAIRS   = None


# =========================
# 2) Listado y pares adyacentes (sin tiempos)
# =========================
def list_images_sorted(frames_dir, exts=(".png",".jpg",".jpeg",".bmp")):
    files = []
    for e in exts:
        files += glob.glob(os.path.join(frames_dir, f"*{e}"))
    files = sorted(files)
    if not files:
        raise RuntimeError(f"No se encontraron imágenes en {frames_dir}")
    return files

def make_pairs_stride(frames_dir, skip=1):
    seq = list_images_sorted(frames_dir)
    pairs = []
    for i in range(len(seq) - skip):
        p_t   = seq[i]
        p_tp  = seq[i + skip]
        pairs.append((p_t, p_tp))
    return pairs


# =========================
# 3) Cargar MMSeg y enganchar features
# =========================
from mmseg.apis import init_model, inference_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model = init_model(CFG_FILE, CKPT_FILE, device=device)
model.eval()

_last_feat = {"x": None}
def _backbone_hook(module, inp, out):
    # 'out' suele ser tupla/lista de features; nos quedamos con FEAT_INDEX
    x = out[FEAT_INDEX] if isinstance(out, (list, tuple)) else out
    if x.dim() == 3:
        x = x.unsqueeze(0)
    _last_feat["x"] = x

handle = model.backbone.register_forward_hook(_backbone_hook)

@torch.no_grad()
def get_pred_and_feat(img_path):
    """Devuelve (pred_np[H,W], feat_tensor[1,C,Hf,Wf])"""
    result = inference_model(model, img_path)  # dispara el hook del backbone
    pred = result.pred_sem_seg.data[0].to('cpu').numpy().astype(np.int32)
    feat = _last_feat["x"]
    return pred, feat


# =========================
# 4) Perceptual Consistency (mem-safe, fixes FP16 y pad)
# =========================
@torch.no_grad()
def pc_pair_memsafe(
    feat_t, feat_tp, pred_t, pred_tp,
    R, drv_id=1,
    Kmax=17, tile=64, use_fp16=True, ds_min=1
):
    """
    Calcula PC entre (t -> t') evitando OOM:
      - Si R > (Kmax-1)//2: avg-pool (ds) y usa R' <= (Kmax-1)//2.
      - Procesa por tiles espaciales.
      - Usa FP16 en features si hay CUDA, pero convierte similitudes a FP32 para enmascarar.
    Devuelve: PC_all, PC_drivable, info
    """
    device_t = feat_t.device
    B, C, Hf, Wf = feat_t.shape
    assert B == 1
    Rmax = (Kmax - 1) // 2

    # downsample adaptativo si R grande
    ds = max(ds_min, int(math.ceil(R / max(1, Rmax)))) if R > Rmax else 1
    R_ = int(math.ceil(R / ds))
    K = 2 * R_ + 1

    # normalización L2 y dtype
    if use_fp16 and device_t.type == "cuda":
        feat_t  = feat_t.half()
        feat_tp = feat_tp.half()
    feat_t  = F.normalize(feat_t,  dim=1)
    feat_tp = F.normalize(feat_tp, dim=1)

    # redimensionar predicciones a tamaño de features
    def _resize_pred(pred_np, W, H):
        return cv2.resize(pred_np, (W, H), interpolation=cv2.INTER_NEAREST)

    feat_t_ds, feat_tp_ds = feat_t, feat_tp
    H2, W2 = Hf, Wf
    if ds != 1:
        feat_t_ds  = F.avg_pool2d(feat_t,  kernel_size=ds, stride=ds, ceil_mode=True)
        feat_tp_ds = F.avg_pool2d(feat_tp, kernel_size=ds, stride=ds, ceil_mode=True)
        H2, W2 = feat_t_ds.shape[-2:]

    pred_t_r  = _resize_pred(pred_t,  W2, H2)
    pred_tp_r = _resize_pred(pred_tp, W2, H2)

    # padding (pred a float para que 'replicate' funcione en CUDA)
    pad = (R_, R_, R_, R_)
    feat_tp_pad = F.pad(feat_tp_ds, pad, mode='replicate')
    pred_tp_pad = F.pad(torch.from_numpy(pred_tp_r).to(device_t).float().view(1,1,H2,W2),
                        pad, mode='replicate')

    sum_r_all, cnt_all = 0.0, 0
    sum_r_drv, cnt_drv = 0.0, 0

    y = 0
    while y < H2:
        th = min(TILE, H2 - y)
        x = 0
        while x < W2:
            tw = min(TILE, W2 - x)

            crop_feat_tp = feat_tp_pad[:, :, y:y+th+2*R_, x:x+tw+2*R_]
            crop_pred_tp = pred_tp_pad[:, :, y:y+th+2*R_, x:x+tw+2*R_]

            patches = F.unfold(crop_feat_tp, kernel_size=K, padding=0, stride=1)  # (1, C*K*K, th*tw)
            patches = patches.view(1, C, K*K, th*tw)

            f_t = feat_t_ds[:, :, y:y+th, x:x+tw].reshape(1, C, th*tw)

            # similitudes en FP32 para evitar overflow en el enmascarado
            sim = (f_t.unsqueeze(2) * patches).sum(dim=1).float()   # (1, K*K, th*tw)
            sim01 = (sim + 1.0) * 0.5                               # [0,1] en FP32

            c_star, _ = sim01.max(dim=1)                            # (1, th*tw)

            pred_t_tile = torch.from_numpy(pred_t_r[y:y+th, x:x+tw]).to(device_t).view(1,1,th*tw)
            pred_unf = F.unfold(crop_pred_tp.float(), kernel_size=K, padding=0, stride=1)  # (1, K*K, th*tw)
            same = (pred_unf.long() == pred_t_tile.long())          # bool

            # enmascarar entradas que no preservan clase con un negativo grande (FP32)
            sim_mask = sim01.masked_fill(~same, -1e6)
            c_dag, _ = sim_mask.max(dim=1)
            c_dag = torch.clamp(c_dag, min=0.0)

            r = (c_dag / (c_star + 1e-6)).view(th*tw)

            sum_r_all += r.sum().item()
            cnt_all   += r.numel()

            keep = (pred_t_tile.view(th*tw) == int(DRIVABLE_ID))
            if keep.any():
                sum_r_drv += r[keep].sum().item()
                cnt_drv   += int(keep.sum().item())

            x += tw
        y += th

    pc_all = (sum_r_all / max(1, cnt_all))
    pc_drv = (sum_r_drv / max(1, cnt_drv)) if cnt_drv > 0 else float('nan')
    return pc_all, pc_drv, {"ds": ds, "R_used": R_, "K": K, "HfWf": (H2, W2)}


# =========================
# 5) Loop principal: pares -> PC -> CSV (R fijo)
# =========================
def run_pc_fixedR(img_dir, out_csv, R_fixed=R_FIXED, skip=1,
                  feat_index=FEAT_INDEX, kmax=KMAX, tile=TILE, use_fp16=USE_FP16, ds_min=DS_MIN,
                  max_pairs=None):
    if R_fixed is None or int(R_fixed) <= 0:
        raise ValueError("R_fixed debe ser un entero positivo (p.ej., 8/12/16/24).")

    # construir pares por stride (sin tiempos)
    pairs = make_pairs_stride(img_dir, skip=skip)
    if max_pairs is not None:
        pairs = pairs[:max_pairs]

    print(f"Pares construidos: {len(pairs)} (primeros 5):")
    for a,b in pairs[:5]:
        print(os.path.basename(a), "→", os.path.basename(b))

    rows = []
    last_path = None
    last_pred = None
    last_feat = None

    for (p_t, p_tp) in pairs:
        # reutilizar si el t actual == último tp
        if p_t == last_path:
            pred_t, feat_t = last_pred, last_feat
        else:
            pred_t, feat_t = get_pred_and_feat(p_t)

        pred_tp, feat_tp = get_pred_and_feat(p_tp)

        pc_all, pc_drv, info = pc_pair_memsafe(
            feat_t, feat_tp, pred_t, pred_tp,
            R=R_fixed, drv_id=DRIVABLE_ID,
            Kmax=kmax, tile=tile, use_fp16=use_fp16, ds_min=ds_min
        )

        rows.append((
            os.path.basename(p_t),
            os.path.basename(p_tp),
            int(R_fixed),
            f"{pc_all:.4f}",
            f"{pc_drv:.4f}",
            f"backbone_out{feat_index}",
            info["ds"], info["R_used"], info["K"], info["HfWf"][0], info["HfWf"][1],
            skip
        ))

        # actualizar caché simple
        last_path = p_tp
        last_pred = pred_tp
        last_feat = feat_tp

        print(rows[-1])  # log

    header = ["img_t","img_tp","R","PC_all","PC_drivable","feat_source",
              "ds","R_used","K","Hf","Wf","skip"]
    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    print(f"\nGuardado: {out_csv}  (filas={len(rows)})")


# =========================
# 6) Ejecutar (R fijo, sin tiempos)
# =========================
run_pc_fixedR(
    IMG_DIR, OUT_CSV,
    R_fixed=R_FIXED,  # <<< controla aquí el R
    skip=SKIP,        # y aquí si quieres saltar frames dentro del directorio
    feat_index=FEAT_INDEX, kmax=KMAX, tile=TILE, use_fp16=USE_FP16, ds_min=DS_MIN,
    max_pairs=MAX_PAIRS
)